# ReformLab Calibration Workflow — Fitting Taste Parameters to Observed Data

**What is calibration?** Before using a discrete choice model to simulate household behaviour, you need to ensure its parameters match reality. The *calibration* process adjusts the β_cost coefficient — the weight households place on cost differences — so that the model's simulated transition rates match observed institutional data (e.g., SDES vehicle fleet surveys).

**What you'll learn:**
1. Understand the calibration target format and load observed transition rates
2. Build a CostMatrix and run the CalibrationEngine to fit β parameters
3. Inspect convergence diagnostics and visualise training fit
4. Validate calibrated parameters against holdout data from a different time period
5. Record calibration provenance in a RunManifest for reproducibility
6. Extract calibrated parameters and use them in a discrete choice simulation

**Prerequisites:** ReformLab installed with calibration and discrete choice modules.

**Related guides:** [08_discrete_choice_model.ipynb](08_discrete_choice_model.ipynb) | [09_population_generation.ipynb](09_population_generation.ipynb)

**Time:** ~30 minutes (reading + execution)

---
## Section 0: Setup

We import the public API for calibration, discrete choice, and governance. All data comes from local fixtures for offline CI execution.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pyarrow as pa

from reformlab.calibration import (
    CalibrationConfig,
    CalibrationEngine,
    CalibrationTarget,
    CalibrationTargetSet,
    capture_calibration_provenance,
    extract_calibrated_parameters,
    load_calibration_targets,
    make_calibration_reference,
    validate_holdout,
)
from reformlab.discrete_choice import (
    CostMatrix,
    TasteParameters,
    compute_probabilities,
    compute_utilities,
)
from reformlab.governance.manifest import RunManifest
from reformlab.visualization import plot_grouped_bars, show_figure

# Resolve paths relative to this notebook
_NB_DIR = Path(__file__).parent if "__file__" in dir() else Path(".")
FIXTURES_DIR = (_NB_DIR / "../../tests/fixtures/calibration").resolve()

SEED = 42  # Deterministic seed for reproducibility

print("✓ Calibration workflow API loaded")
print(f"✓ Fixtures directory: {FIXTURES_DIR}")


---
## Section 1: Load Calibration Targets

**What is a calibration target?** A calibration target records the observed fraction of households in a given `from_state` that chose a given `to_state` during a reference period, according to institutional data.

Each row has six fields:
| Field | Example | Meaning |
|-------|---------|----------|
| `domain` | `vehicle` | Decision domain (vehicle, heating, …) |
| `period` | `2022` | Reference year for the observed data |
| `from_state` | `petrol` | Current household state (origin) |
| `to_state` | `buy_electric` | Chosen alternative (destination) |
| `observed_rate` | `0.03` | Fraction of households that made this choice |
| `source_label` | `SDES 2022` | Human-readable data source identifier |

Real production data comes from sources like ADEME vehicle adoption surveys and SDES fleet statistics. Here we use a fixture file that illustrates the full format with four vehicle alternatives.

In [ ]:
# Load the production-format training targets from fixture CSV
# This file contains 6 targets across 2 from_states and 4 to_states
target_set = load_calibration_targets(FIXTURES_DIR / "valid_vehicle_targets.csv")

print(f"Loaded {len(target_set.targets)} calibration targets from valid_vehicle_targets.csv")
print()
print(f"{'domain':<10} {'period':<8} {'from_state':<12} {'to_state':<18} {'observed_rate':<15} {'source_label'}")
print("-" * 85)
for t in target_set.targets:
    print(f"{t.domain:<10} {t.period:<8} {t.from_state:<12} {t.to_state:<18} {t.observed_rate:<15.3f} {t.source_label}")

**Reading the output:**
- Two `from_state` groups: `petrol` (4 alternatives, rates sum to 0.95) and `ev` (2 alternatives, rates sum to 0.97). Rates may sum to less than 1.0 — the remainder represents households not yet classified.
- The data source is `SDES vehicle fleet 2022`, a French government fleet statistics publication.

**For this notebook demo** we use a simplified 2-alternative model (alternatives `A` and `B`) matching the conftest pattern, which is easier to reason about and visualise than the full 4-alternative structure. The calibration mechanics are identical regardless of the number of alternatives.

---
## Section 2: Build Cost Matrix

**What is a CostMatrix?** A CostMatrix is an N×M table where N = number of households and M = number of alternatives. Each cell [i, j] = cost for household i choosing alternative j.

Cost typically represents the total cost of ownership (purchase price, fuel, maintenance) or some composite measure. The calibration engine uses these costs to compute utility differences via:

$$V_{ij} = \beta_{cost} \times cost_{ij}$$

A more negative β_cost means households are more sensitive to cost differences.

In [ ]:
# Training population: 3 households, 2 alternatives (A = cheaper EV, B = petrol)
# Costs are notional (EUR/year total cost of ownership)
training_cost_matrix = CostMatrix(
    table=pa.table({
        "A": pa.array([100.0, 150.0, 200.0]),
        "B": pa.array([200.0, 100.0, 300.0]),
    }),
    alternative_ids=("A", "B"),
)

# from_states: which vehicle type each household currently owns
training_from_states = pa.array(["petrol", "petrol", "diesel"])

print(f"Training CostMatrix: {training_cost_matrix.n_households} households × {training_cost_matrix.n_alternatives} alternatives")
print(f"Alternative IDs: {training_cost_matrix.alternative_ids}")
print()
print("Cost matrix (EUR/year):")
print(f"  {'Household':<12} {'from_state':<12} {'Cost A':>10} {'Cost B':>10}")
print(f"  {'-'*46}")
cost_a = training_cost_matrix.table.column("A").to_pylist()
cost_b = training_cost_matrix.table.column("B").to_pylist()
from_list = training_from_states.to_pylist()
for i, (fs, ca, cb) in enumerate(zip(from_list, cost_a, cost_b)):
    print(f"  H{i:<11} {fs:<12} {ca:>10.0f} {cb:>10.0f}")

**Reading the cost matrix:**
- H0 (petrol): alternative A costs 100, B costs 200 → A is clearly cheaper
- H1 (petrol): alternative A costs 150, B costs 100 → B is cheaper
- H2 (diesel): alternative A costs 200, B costs 300 → A is cheaper

The `from_state` tells the calibration engine which observed transition rates apply to each household. H0 and H1 are matched to petrol targets; H2 is matched to diesel targets.

---
## Section 3: Run Calibration Engine

**What does the calibration engine do?** It iteratively adjusts β_cost until the model's simulated transition rates match the observed rates from institutional data. At each iteration:

1. Compute utilities: $V_{ij} = \beta \times cost_{ij}$
2. Compute choice probabilities via the logit model
3. Aggregate probabilities by from_state to get simulated rates
4. Evaluate objective (MSE between observed and simulated rates)
5. Adjust β and repeat until convergence

The optimisation is **deterministic** — same inputs always produce the same result. We use L-BFGS-B (a gradient-based method) from scipy.optimize.

In [ ]:
# Build simplified training targets matching the 2-alternative cost matrix
# Observed rates from a 2022 survey of 3 household archetypes:
#   petrol households: 40% chose A, 55% chose B (5% not recorded)
#   diesel households: 60% chose A (40% not recorded)
training_target_set = CalibrationTargetSet(
    targets=(
        CalibrationTarget(
            domain="vehicle", period=2022,
            from_state="petrol", to_state="A",
            observed_rate=0.40, source_label="SDES 2022 demo",
        ),
        CalibrationTarget(
            domain="vehicle", period=2022,
            from_state="petrol", to_state="B",
            observed_rate=0.55, source_label="SDES 2022 demo",
        ),
        CalibrationTarget(
            domain="vehicle", period=2022,
            from_state="diesel", to_state="A",
            observed_rate=0.60, source_label="SDES 2022 demo",
        ),
    )
)

print(f"Training target set: {len(training_target_set.targets)} targets")
for t in training_target_set.targets:
    print(f"  {t.from_state} \u2192 {t.to_state}: observed_rate={t.observed_rate:.2f}")

In [ ]:
# Configure and run the calibration engine
config = CalibrationConfig(
    targets=training_target_set,
    cost_matrix=training_cost_matrix,
    from_states=training_from_states,
    domain="vehicle",
    initial_beta=-0.01,
    objective_type="mse",
    method="L-BFGS-B",
    max_iterations=100,
    rate_tolerance=0.05,
)

engine = CalibrationEngine(config=config)
result = engine.calibrate()

print("Calibration complete!")
print()
print(f"{'Diagnostic':<30} {'Value'}")
print("-" * 50)
print(f"{'Optimised \u03b2_cost':<30} {result.optimized_parameters.beta_cost:.6f}")
print(f"{'Convergence flag':<30} {result.convergence_flag}")
print(f"{'Iterations':<30} {result.iterations}")
print(f"{'Gradient norm':<30} {result.gradient_norm}")
print(f"{'Method':<30} {result.method}")
print(f"{'Objective type':<30} {result.objective_type}")
print(f"{'Final objective value':<30} {result.objective_value:.8f}")
print(f"{'All within tolerance':<30} {result.all_within_tolerance}")

In [ ]:
# Per-target rate comparisons
print("Per-target rate comparisons:")
print()
print(f"  {'from_state':<12} {'to_state':<10} {'observed':>10} {'simulated':>10} {'abs_error':>10} {'within_tol':>12}")
print(f"  {'-'*57}")
for rc in result.rate_comparisons:
    print(
        f"  {rc.from_state:<12} {rc.to_state:<10} "
        f"{rc.observed_rate:>10.4f} {rc.simulated_rate:>10.4f} "
        f"{rc.absolute_error:>10.4f} {str(rc.within_tolerance):>12}"
    )

**Interpreting convergence diagnostics:**
- **convergence_flag=True** means the L-BFGS-B optimizer satisfied its stopping criteria (gradient norm below tolerance)
- **gradient_norm** near zero confirms we found a local minimum of the MSE objective
- **all_within_tolerance** checks whether each simulated rate is within ±0.05 of the observed rate
- The optimised **β_cost** is typically negative — households prefer lower-cost alternatives

The simulated rates may not perfectly match observed rates because: (1) the small population (3 households) limits statistical precision; (2) the logit model enforces probability normalization across alternatives.

---
## Section 4: Visualize Training Fit

A bar chart makes it easy to see at a glance how well the calibrated model matches each observed rate. We compare `observed_rate` vs `simulated_rate` per target, and mark the tolerance threshold.

In [ ]:
target_labels = [f"{rc.from_state} → {rc.to_state}" for rc in result.rate_comparisons]
fig, ax = plot_grouped_bars(
    target_labels,
    {
        "Observed": [rc.observed_rate for rc in result.rate_comparisons],
        "Simulated": [rc.simulated_rate for rc in result.rate_comparisons],
    },
    title="Training: Observed vs Simulated Transition Rates",
    xlabel="Transition (from_state → to_state)",
    ylabel="Rate",
    colors=["#2196F3", "#FF9800"],
    xtick_rotation=15,
    ylim=(0, 1.0),
)

# Show tolerance bands around each observed rate (±tolerance)
import numpy as np

observed_rates = [rc.observed_rate for rc in result.rate_comparisons]
x_positions = np.arange(len(observed_rates))
tol = config.rate_tolerance

for i, obs in enumerate(observed_rates):
    ax.fill_between(
        [i - 0.4, i + 0.4],
        obs - tol, obs + tol,
        color="red", alpha=0.10, zorder=0,
    )
    ax.hlines(
        [obs - tol, obs + tol], i - 0.4, i + 0.4,
        colors="red", linestyles="--", alpha=0.5, linewidths=0.8,
    )

# Single legend entry for the tolerance band
from matplotlib.patches import Patch
ax.legend(
    handles=[*ax.get_legend_handles_labels()[0],
             Patch(facecolor="red", alpha=0.15, edgecolor="red",
                   linestyle="--", label=f"±{tol} tolerance")],
)
show_figure(fig)

print(f"All within tolerance (±{config.rate_tolerance}): {result.all_within_tolerance}")

**Reading the chart:**
- Blue bars = observed rates from institutional data
- Orange bars = simulated rates from the calibrated model
- Red shaded bands = ±`rate_tolerance` (0.05) around each observed rate

A target is *within tolerance* if the simulated bar falls inside the red band for that target. The goal is `all_within_tolerance = True`. If some targets fall outside, consider: adjusting `rate_tolerance`, using a larger population, or switching to the `log_likelihood` objective.

---
## Section 5: Holdout Validation

**Why validate on holdout data?** Training fit tells you how well the model fits the data it was calibrated on. Holdout validation asks: *does the calibrated β generalise to a different time period?*

We use 2023 transition data (the holdout period) to validate a model calibrated on 2022 data. If the holdout fit is similar to the training fit, the calibrated β is likely stable. A much worse holdout fit suggests overfitting or a structural break.

In [ ]:
# Holdout population: same 3 households, slightly different 2023 costs
holdout_cost_matrix = CostMatrix(
    table=pa.table({
        "A": pa.array([110.0, 140.0, 210.0]),
        "B": pa.array([190.0, 110.0, 280.0]),
    }),
    alternative_ids=("A", "B"),
)
holdout_from_states = pa.array(["petrol", "petrol", "diesel"])

# Load holdout targets from fixture CSV (period=2023, slightly different rates)
holdout_target_set = load_calibration_targets(FIXTURES_DIR / "holdout_vehicle_targets.csv")

print(f"Holdout CostMatrix: {holdout_cost_matrix.n_households} households")
print(f"Holdout targets: {len(holdout_target_set.targets)} targets (period=2023)")
print()
for t in holdout_target_set.targets:
    print(f"  {t.from_state} \u2192 {t.to_state}: observed_rate={t.observed_rate:.2f} (period={t.period})")

In [ ]:
# Validate calibrated parameters on holdout data
holdout_result = validate_holdout(
    result,
    holdout_target_set,
    holdout_cost_matrix,
    holdout_from_states,
    rate_tolerance=config.rate_tolerance,
)

print(f"Holdout validation complete for domain={holdout_result.domain!r}")

In [ ]:
# Side-by-side training vs holdout fit metrics
print(f"{'Metric':<25} {'Training':>12} {'Holdout':>12}")
print("-" * 50)
print(f"{'MSE':<25} {holdout_result.training_fit.mse:>12.6f} {holdout_result.holdout_fit.mse:>12.6f}")
print(f"{'MAE':<25} {holdout_result.training_fit.mae:>12.6f} {holdout_result.holdout_fit.mae:>12.6f}")
print(f"{'Targets':<25} {holdout_result.training_fit.n_targets:>12d} {holdout_result.holdout_fit.n_targets:>12d}")
print(f"{'All Within Tolerance':<25} {str(holdout_result.training_fit.all_within_tolerance):>12} {str(holdout_result.holdout_fit.all_within_tolerance):>12}")
print()
print("Holdout per-target comparisons:")
for rc in holdout_result.holdout_rate_comparisons:
    print(f"  {rc.from_state} \u2192 {rc.to_state}: obs={rc.observed_rate:.3f} sim={rc.simulated_rate:.3f} err={rc.absolute_error:.3f}")

**Interpreting the side-by-side metrics:**
- **MSE** (Mean Squared Error) and **MAE** (Mean Absolute Error) both measure calibration quality. Lower is better.
- Good generalisation: holdout MSE is close to training MSE. The β coefficient is stable across the 2022→2023 shift.
- Poor generalisation: holdout MSE much larger than training MSE. Possible structural break (e.g., subsidy introduction) or overfitting to training noise.

The validate_holdout result is **non-blocking** — a poor holdout fit raises a warning, not an exception. You decide how to react.

---
## Section 6: Governance Provenance

**Why record provenance?** Every calibration run produces a β coefficient that will influence downstream simulations. For reproducibility and policy credibility, the exact inputs and diagnostics must be recorded in a run manifest. `capture_calibration_provenance()` aggregates all relevant entries in one call.

In [ ]:
# Capture all calibration governance entries
entries = capture_calibration_provenance(
    result,
    target_set=training_target_set,
    holdout_result=holdout_result,
)

print(f"Governance entries captured: {len(entries)}")
print()
for entry in entries:
    print(f"Key: {entry['key']}")
    print(f"  source     : {entry['source']}")
    print(f"  is_default : {entry['is_default']}")
    value = entry['value']
    if isinstance(value, dict):
        for k, v in value.items():
            print(f"  {k:<28}: {v}")
    print()

In [ ]:
# Create a calibration reference entry (links this simulation back to the calibration run)
ref_entry = make_calibration_reference("calib-run-2026-03-07-v1")
print("Calibration reference entry:")
print(f"  key        : {ref_entry['key']}")
print(f"  value      : {ref_entry['value']}")
print(f"  source     : {ref_entry['source']}")
print()

# Combine calibration entries + reference entry for a complete manifest
manifest_assumptions = [*entries, ref_entry]
manifest = RunManifest(
    manifest_id="demo-manifest-2026-03-07",
    created_at="2026-03-07T00:00:00",
    engine_version="0.1.0",
    openfisca_version="44.0.0",
    adapter_version="0.1.0",
    scenario_version="v1",
    assumptions=manifest_assumptions,
)
print(f"RunManifest created with {len(manifest.assumptions)} assumption entries")
print(f"Assumption keys: {[a['key'] for a in manifest.assumptions]}")

**Governance entry keys:**
- `calibration_result` — the optimised β, convergence diagnostics, and per-target fit quality
- `calibration_targets` — metadata about the training dataset (domains, periods, n_targets, sources)
- `holdout_validation` — side-by-side training/holdout MSE and MAE for reproducibility
- `calibration_reference` (optional) — traceability link from a simulation manifest back to its calibration manifest

In a production workflow, the manifest is saved to disk and hashed. Any downstream analyst can reproduce the exact calibrated β from the manifest alone.

---
## Section 7: Parameter Round-Trip

**Why is round-trip extraction important?** When a simulation run uses calibrated parameters from a prior calibration run, the β must be recovered exactly from the manifest — not re-calibrated. `extract_calibrated_parameters()` performs this extraction.

This enables the calibration → manifest → simulation lifecycle: calibrate once, record in manifest, extract for any future simulation.

In [ ]:
# Extract calibrated TasteParameters from the governance entries
extracted = extract_calibrated_parameters(entries, "vehicle")

print(f"Extracted TasteParameters:")
print(f"  extracted.beta_cost  = {extracted.beta_cost:.8f}")
print(f"  original  beta_cost  = {result.optimized_parameters.beta_cost:.8f}")
print()

# Verify exact equality (the manifest preserves the exact float value)
assert extracted.beta_cost == result.optimized_parameters.beta_cost, (
    f"Round-trip mismatch: {extracted.beta_cost} != {result.optimized_parameters.beta_cost}"
)
print("\u2713 Exact equality confirmed: manifest round-trip preserves beta_cost perfectly")

# Show how to reconstruct a TasteParameters from scratch if you know the value
reconstructed = TasteParameters(beta_cost=extracted.beta_cost)
assert reconstructed.beta_cost == extracted.beta_cost
print(f"\u2713 TasteParameters({reconstructed.beta_cost:.6f}) reconstructed successfully")

**Why this matters:**
- The calibration run and the simulation run can happen days or weeks apart — even on different machines
- The manifest contains the exact float value of β, enabling deterministic reconstruction
- `extract_calibrated_parameters()` validates that the entry exists, is not duplicated, and has a numeric β value — raising `CalibrationProvenanceError` for corrupted manifests

---
## Section 8: Using Calibrated Parameters in Simulation

The final step is using the calibrated β to run the discrete choice model. We pass the extracted `TasteParameters` to `compute_utilities()` and `compute_probabilities()` — the same functions used inside the orchestrator pipeline.

In [ ]:
# Use extracted TasteParameters with the training CostMatrix
utilities = compute_utilities(training_cost_matrix, extracted)
probabilities = compute_probabilities(utilities)

print(f"Calibrated \u03b2_cost = {extracted.beta_cost:.6f}")
print()
print("Per-household choice probability matrix:")
print()
print(f"  {'Household':<12} {'from_state':<12} {'P(choose A)':>12} {'P(choose B)':>12}")
print(f"  {'-'*50}")
prob_a = probabilities.column("A").to_pylist()
prob_b = probabilities.column("B").to_pylist()
for i, (fs, pa_val, pb_val) in enumerate(zip(from_list, prob_a, prob_b)):
    print(f"  H{i:<11} {fs:<12} {pa_val:>12.4f} {pb_val:>12.4f}")

print()
print("Verification: all probability rows sum to 1.0")
for i, (pa_val, pb_val) in enumerate(zip(prob_a, prob_b)):
    row_sum = pa_val + pb_val
    assert abs(row_sum - 1.0) < 1e-10, f"H{i}: probability sum = {row_sum}"
    print(f"  H{i}: P(A) + P(B) = {pa_val:.4f} + {pb_val:.4f} = {row_sum:.4f} \u2713")

In [ ]:
# Visualise the per-household probability matrix
household_labels = [f"H{i} ({fs})" for i, fs in enumerate(from_list)]
fig, ax = plot_grouped_bars(
    household_labels,
    {
        "P(choose A)": prob_a,
        "P(choose B)": prob_b,
    },
    title=f"Per-Household Choice Probabilities (calibrated β_cost = {extracted.beta_cost:.4f})",
    xlabel="Household",
    ylabel="Choice Probability",
    colors=["#4CAF50", "#E91E63"],
    ylim=(0, 1.0),
)
show_figure(fig)

print("✓ Calibrated β feeds correctly through the discrete choice pipeline")


**Reading the probability matrix:**
- H0 (petrol): A costs 100, B costs 200 → H0 strongly prefers A
- H1 (petrol): A costs 150, B costs 100 → H1 strongly prefers B
- H2 (diesel): A costs 200, B costs 300 → H2 prefers A (less strongly, since both are expensive)

The calibrated β_cost determines how sensitive households are to these cost differences. A more negative β produces steeper probability differences between alternatives.

---
## Section 9: Summary and Next Steps

### Calibration Workflow Recap

You've completed the full calibration lifecycle:

| Step | Action | Output |
|------|--------|--------|
| 1 | Load calibration targets | CalibrationTargetSet (observed rates) |
| 2 | Build cost matrix | CostMatrix (N households × M alternatives) |
| 3 | Run calibration engine | CalibrationResult (optimised β + diagnostics) |
| 4 | Visualise training fit | Observed vs simulated rate bar chart |
| 5 | Validate on holdout data | HoldoutValidationResult (training vs holdout MSE) |
| 6 | Record governance provenance | List of AssumptionEntry dicts → RunManifest |
| 7 | Extract parameters | TasteParameters (exact β from manifest) |
| 8 | Run simulation | Choice probabilities via compute_utilities + compute_probabilities |

### Adapting for Production

- **Larger populations**: Replace the 3-household fixture with a full synthetic population from `09_population_generation.ipynb`. Calibration quality improves significantly with 1,000+ households.
- **Real institutional data**: Replace fixture CSVs with ADEME vehicle adoption data or SDES fleet statistics loaded via `load_calibration_targets()`.
- **Multiple domains**: Run separate CalibrationEngine instances for vehicle and heating domains, then combine the provenance entries in a single RunManifest.
- **Log-likelihood objective**: Set `objective_type='log_likelihood'` in CalibrationConfig for statistically principled parameter estimation.

### Related Guides

- [08_discrete_choice_model.ipynb](08_discrete_choice_model.ipynb) — Full 10-year behavioural simulation with calibrated parameters
- [09_population_generation.ipynb](09_population_generation.ipynb) — Generating synthetic French household populations
- [05_governance_reproducibility.ipynb](05_governance_reproducibility.ipynb) — Full RunManifest lifecycle and integrity checking